# ST-01 — Siretisation Phase 1 (SIRET exact)

Pour chaque EG FINESS ayant un `nmsiret_stru`, lookup direct dans la base Etab SIRENE complète. Calcul du score (nom + adresse) et classification.

Statuts produits : VALIDE_FORT / VALIDE / DOUTEUX / REJETE / SANS_SIRET / SIRET_INCONNU

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.siretisation import matching_direct_siret
from src.excel_export import export_phase1_excel, LABELS
from src.display      import afficher_tableau, afficher_synthese
from config.settings  import (
    FINESS_EG_CLEAN, SIRENE_ETAB_CLEAN, ST_PHASE1, RESULTS_ST_DIR,
)

RESULTS_ST_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement

In [2]:
df_eg   = pd.read_parquet(FINESS_EG_CLEAN)
df_etab = pd.read_parquet(SIRENE_ETAB_CLEAN)

df_eg['nmsiret_stru'] = df_eg['nmsiret_stru'].fillna('').astype(str)
df_etab['siret']      = df_etab['siret'].astype(str)

print(f'EG FINESS  : {len(df_eg):,}')
print(f'Etab SIRENE: {len(df_etab):,}')

EG FINESS  : 104,612
Etab SIRENE: 16,867,946


## 2. Matching SIRET exact

In [3]:
df_resultats = matching_direct_siret(df_eg, df_etab, desc='Matching SIRET exact')
print(df_resultats['statut'].value_counts())

Matching SIRET exact:   0%|          | 0/104612 [00:00<?, ?it/s]

statut
VALIDE           38121
VALIDE_FORT      21605
SANS_SIRET       13129
SIRET_INCONNU    12198
DOUTEUX          10615
REJETE            8944
Name: count, dtype: int64


## 3. Enrichissement avec colonnes Etab SIRENE

In [4]:
COLS_ETAB_JOIN = ['siret', 'denominationUniteLegale', 'sigleUniteLegale',
                  'enseigne1Etablissement', 'enseigne2Etablissement',
                  'enseigne3Etablissement', 'denominationUsuelleEtablissement',
                  'adresse_complete_etab', 'codeCommuneEtablissement',
                  'categorieJuridiqueUniteLegale', 'activitePrincipaleUniteLegale']
etab_join = df_etab[[c for c in COLS_ETAB_JOIN if c in df_etab.columns]].drop_duplicates('siret').copy()
etab_join['siret'] = etab_join['siret'].astype(str)

df_resultats['siret_etab'] = df_resultats['siret_etab'].astype(str)
df_resultats = df_resultats.merge(
    etab_join, left_on='siret_etab', right_on='siret', how='left',
).drop(columns=['siret'], errors='ignore')

## 4. Aperçu

In [5]:
afficher_tableau(
    df_resultats[df_resultats['statut'].isin(['VALIDE_FORT', 'VALIDE'])],
    'Aperçu validés', max_lignes=9,
    colonnes=['idstructure_stru', 'raisonsociale_stru',
              'denominationUniteLegale', 'nom_etab_retenu',
              'score_nom', 'score_adresse', 'score_global', 'statut'],
)

idstructure_stru,raisonsociale_stru,denominationUniteLegale,nom_etab_retenu,score_nom,score_adresse,score_global,statut
1931885,PHARMACIE DE VESONE,PHARMACIE DE VESONE,PHARMACIE VESONE,100.000000,90.590000,94.350000,VALIDE_FORT
1931887,PHARMACIE FENELON,PHARMACIE FENELON,PHARMACIE FENELON,100.000000,87.330000,92.400000,VALIDE_FORT
1931891,PHARMACIE CHAPARD,PHARMACIE CHAPARD,PHARMACIE CHAPARD,100.000000,87.150000,92.290000,VALIDE_FORT
1931896,PHARMACIE DU PALAIS,PHARMACIE DU PALAIS,PHARMACIE PALAIS,100.000000,77.540000,86.520000,VALIDE_FORT
1931899,PHARMACIE DENY-FANTHOU,EURL PHARMACIE DU TOULON,PHARMACIE TOULON,46.630000,90.500000,72.950000,VALIDE
1931900,PHARMACIE SAINT-GEORGES,PHARMACIE ST GEORGES,PHARMACIE ST GEORGES,85.490000,83.560000,84.330000,VALIDE
1931902,PHARMACIE DES BARRIS,PHARMACIE DES BARRIS,PHARMACIE BARRIS,100.000000,100.000000,100.000000,VALIDE_FORT
1931903,PHARMACIE ALIENOR,PHARMACIE ALIENOR,PHARMACIE ALIENOR,100.000000,100.000000,100.000000,VALIDE_FORT
1931908,PHARMACIE DU PÉRIGORD VERT,PHARMACIE DU PERIGORD VERT,PHARMACIE PERIGORD VERT,100.000000,86.670000,92.000000,VALIDE_FORT


## 5. Export Excel

In [6]:
COLS_COMPLET = [
    'idstructure_stru', 'nmfinessej_stru', 'nmfinessetab_stru', 'categetab_stru',
    'nmsiret_stru', 'raisonsociale_stru',
    'cdcommune_stru', 'adresse_complete_eg',
    'siret_etab', 'denominationUniteLegale', 'sigleUniteLegale',
    'enseigne1Etablissement', 'enseigne2Etablissement', 'enseigne3Etablissement',
    'denominationUsuelleEtablissement', 'nom_etab_retenu',
    'adresse_complete_etab', 'codeCommuneEtablissement',
    'categorieJuridiqueUniteLegale', 'activitePrincipaleUniteLegale',
    'score_nom', 'score_adresse', 'score_global',
]
COLS_INFO = [
    'idstructure_stru', 'nmfinessej_stru', 'nmfinessetab_stru', 'categetab_stru',
    'nmsiret_stru', 'raisonsociale_stru',
    'cdcommune_stru', 'adresse_complete_eg',
]

compteurs = export_phase1_excel(
    df_resultats, ST_PHASE1, COLS_COMPLET, COLS_INFO,
    statuts_score=['VALIDE_FORT', 'VALIDE', 'DOUTEUX', 'REJETE'],
    statuts_info =['SANS_SIRET', 'SIRET_INCONNU'],
)

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  'Synthèse Siretisation Phase 1')
print(f'\nFichier : {ST_PHASE1}')

Statut,Nb,% du total
Valide_fort,"21,605",20.7%
Valide,"38,121",36.4%
Douteux,"10,615",10.1%
Rejeté,"8,944",8.5%
Sans_SIRET,"13,129",12.6%
SIRET_inconnu,"12,198",11.7%
TOTAL,"104,612",100.0%



Fichier : /home/jovyan/work/projet_finess_sirene/results/siretisation/siretisation_phase1.xlsx
